In [6]:
import cv2
import numpy as np
import os
from itertools import permutations

BORDER_WIDTH = 3
MSE_THRESHOLD = 300
OUT_DIR = "./results/2x2_out"


# LOAD PIECES
def load_pieces(folder):
    def key(name):
        digits = ''.join(c for c in name if c.isdigit())
        return int(digits) if digits else 0

    files = sorted(
        [f for f in os.listdir(folder) if f.lower().endswith(('.png','.jpg'))],
        key=key
    )

    pieces = []
    for f in files:
        img = cv2.imread(os.path.join(folder, f))
        if img is not None:
            pieces.append(img)
    return pieces


# BORDER EXTRACTION

def border(img, side):
    w = BORDER_WIDTH
    if side == 'top':    return img[:w]
    if side == 'bottom': return img[-w:]
    if side == 'left':   return img[:, :w]
    if side == 'right':  return img[:, -w:]



# EDGE FEATURES
def edge_features(edge):
    edge = cv2.GaussianBlur(edge, (3,3), 0)

    lab  = cv2.cvtColor(edge, cv2.COLOR_BGR2LAB).astype(np.float32)
    gray = cv2.cvtColor(edge, cv2.COLOR_BGR2GRAY).astype(np.float32)

    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)

    mag = np.sqrt(gx**2 + gy**2)[...,None]
    lap = cv2.Laplacian(gray, cv2.CV_32F)[...,None]

    return np.concatenate([lab, mag, lap], axis=2)


def normalize(f):
    f = f.copy()
    for c in range(f.shape[2]):
        m, s = f[...,c].mean(), f[...,c].std()
        f[...,c] = (f[...,c]-m)/(s if s > 1e-6 else 1)
    return f


# EDGE DISTANCE

def edge_distance(a, b, side_a, side_b):
    a = normalize(a)
    b = normalize(b)

    if side_a in ('left','right'):
        a = np.transpose(a, (1,0,2))
    if side_b in ('left','right'):
        b = np.transpose(b, (1,0,2))

    if a.shape != b.shape:
        b = cv2.resize(b, (a.shape[1], a.shape[0]))

    d = np.abs(a - b)

    return (
        0.5 * d[...,0:3].mean() +   # LAB
        0.3 * d[...,3].mean()   +   # gradient
        0.2 * d[...,4].mean()       # laplacian
    )



# BUILD COST TABLES

def build_costs(pieces):
    sides = ['top','bottom','left','right']
    opp   = {'top':'bottom','bottom':'top','left':'right','right':'left'}
    n = len(pieces)

    feats = {}
    for i,p in enumerate(pieces):
        for s in sides:
            feats[(i,s)] = edge_features(border(p,s))

    cost = {s: np.full((n,n), np.inf) for s in sides}

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            for s in sides:
                cost[s][i,j] = edge_distance(
                    feats[(i,s)],
                    feats[(j,opp[s])],
                    s, opp[s]
                )
    return cost


# BRUTE FORCE SOLVER 

def solve_2x2(pieces):
    cost = build_costs(pieces)

    best, second = None, 1e18
    best_score = 1e18

    for perm in permutations(range(4)):
        score = (
            cost['right'][perm[0], perm[1]] +
            cost['bottom'][perm[0], perm[2]] +
            cost['right'][perm[2], perm[3]] +
            cost['bottom'][perm[1], perm[3]]
        )

        if score < best_score:
            second = best_score
            best_score = score
            best = perm
        elif score < second:
            second = score

    return list(best), second - best_score


# ASSEMBLE IMAGE


def assemble(pieces, order):
    h, w = pieces[0].shape[:2]
    out = np.zeros((2*h,2*w,3), np.uint8)

    out[0:h,0:w]       = pieces[order[0]]
    out[0:h,w:2*w]     = pieces[order[1]]
    out[h:2*h,0:w]     = pieces[order[2]]
    out[h:2*h,w:2*w]   = pieces[order[3]]

    return out





def mse(a,b):
    if a.shape != b.shape:
        b = cv2.resize(b,(a.shape[1],a.shape[0]))
    return np.mean((a.astype(np.float32)-b.astype(np.float32))**2)


def evaluate(puzzle_dir, gt_dir):
    os.makedirs(OUT_DIR, exist_ok=True)

    folders = sorted(os.listdir(puzzle_dir))
    correct = total = 0

    print("-"*50)
    for f in folders:
        pieces = load_pieces(os.path.join(puzzle_dir,f))
        if len(pieces) != 4:
            continue

        order, margin = solve_2x2(pieces)
        result = assemble(pieces, order)

        # SAVE OUTPUT IMAGE
        cv2.imwrite(os.path.join(OUT_DIR, f"{f}_assembled.png"), result)

        gt = cv2.imread(os.path.join(gt_dir, f+".png"))
        err = mse(result, gt)

        ok = err < MSE_THRESHOLD
        correct += ok
        total += 1

        print(f"{f:<6} | MSE {err:7.1f} | {'PASS' if ok else 'FAIL'} | margin {margin:.2f}")

    print("-"*50)
    print(f"ACCURACY: {correct}/{total} ({100*correct/total:.2f}%)")


 

if __name__ == "__main__":
    evaluate(
        "./results/enhanced_images_sliced/puzzle_2x2",
        "../data/correct"
    )


--------------------------------------------------
0      | MSE   125.3 | PASS | margin 0.38
1      | MSE    31.6 | PASS | margin 0.54
10     | MSE    47.3 | PASS | margin 0.19
100    | MSE    63.7 | PASS | margin 0.76
101    | MSE    51.4 | PASS | margin 0.47
102    | MSE    62.6 | PASS | margin 0.57
103    | MSE   109.8 | PASS | margin 0.54
104    | MSE    91.4 | PASS | margin 0.45
105    | MSE    70.1 | PASS | margin 0.07
106    | MSE    91.8 | PASS | margin 0.70
107    | MSE    64.0 | PASS | margin 0.54
108    | MSE    22.1 | PASS | margin 0.84
109    | MSE    83.1 | PASS | margin 0.80
11     | MSE    60.2 | PASS | margin 0.74
12     | MSE    45.4 | PASS | margin 0.45
13     | MSE    72.8 | PASS | margin 0.47
14     | MSE   112.5 | PASS | margin 0.55
15     | MSE   118.6 | PASS | margin 0.70
16     | MSE    34.3 | PASS | margin 0.98
17     | MSE    49.8 | PASS | margin 0.39
18     | MSE    45.8 | PASS | margin 0.38
19     | MSE    58.1 | PASS | margin 0.78
2      | MSE    61.0 | PA